# S11b 연도·위원회 표기 차이 추출 측정

목적: S11b에서 더한 규칙(`year_ref`, 두 자리 연도, 위원회 약칭 표)을 G3와 dev single-doc에서 한 번 잰다(`docs/slices/11b-condition-variants.md`). 규칙은 G3를 보지 않고 정했다. 정답 문서 메타데이터는 측정에서만 기대값으로 쓰고 검색 점수는 보지 않는다. 질의 원문은 몇 건만 보인다(D-10).

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

from rag.conditions import DATE, Extractor, doc_values
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
CORPUS = ROOT / "data/processed/corpus_context.jsonl"
FROZEN = ROOT / "data/multidoc/queries_g3.jsonl"
QUERIES = ROOT / "data/processed/queries_summary_q.jsonl"
DEV = ROOT / "data/processed/queries_summary_q.dev.jsonl"
META_DICT = ROOT / "data/processed/meta_dict.json"

## 데이터 로딩

In [3]:
with CORPUS.open(encoding="utf-8") as f:
    values = {d["doc_id"]: doc_values(d) for d in map(json.loads, f)}
frozen = {q["qid"]: q for q in map(json.loads, FROZEN.open(encoding="utf-8"))}
dev_qids = {json.loads(line)["qid"] for line in DEV.open(encoding="utf-8")}
single = {q["qid"]: q for q in map(json.loads, QUERIES.open(encoding="utf-8"))
          if q["qid"] in dev_qids}
gold = {"multidoc": {q: (r["gold_doc_ids"], r["type"]) for q, r in frozen.items()},
        "summary_q": {q: ([r["doc_id"]], "single") for q, r in single.items()}}
text = {**{q: r["query"] for q, r in frozen.items()}, **{q: r["query"] for q, r in single.items()}}
print(len(values), {k: len(v) for k, v in gold.items()})

38516 {'multidoc': 600, 'summary_q': 3016}


In [4]:
cfg = load_cfg()
c = cfg.conditions
meta = json.loads(META_DICT.read_text(encoding="utf-8"))
args = (meta, cfg.ingest.name_titles, c.name_suffixes, set(c.name_exclude))
ex_new = Extractor(*args, c.committee_aliases)
ex_old = Extractor(*args)  # 약칭 표 없음(S11)
cond = {q: ex_new.extract(t) for q, t in text.items()}
len(cond)

3616

## 위원회 약칭 표

약칭 표 때문에 위원회 값이 더해진 질의와 그 값의 정밀도·적중률.

In [10]:
rows = []
for name, g in gold.items():
    for qid, (docs, qtype) in g.items():
        old = set(ex_old.extract(text[qid]).get("committee_name", []))
        new = cond[qid].get("committee_name", [])
        added = [v for v in new if v not in old]
        ok = [v for v in added if any(v in values[d]["committee_name"] for d in docs)]
        hit_old = any(v in values[d]["committee_name"] for v in old for d in docs)
        rows.append({"set": name, "type": qtype, "added": len(added), "added_ok": len(ok),
                     "new_hit": bool(ok) and not hit_old})
alias = pd.DataFrame(rows)
alias.groupby(["set", "type"]).agg(
    질의=("added", "size"), 더해진질의=("added", lambda s: (s > 0).sum()),
    더해진값=("added", "sum"), 맞은값=("added_ok", "sum"), 새로적중=("new_hit", "sum"))

질의  더해진질의  더해진값  맞은값  새로적중
set       type                                    
multidoc  conf         233      4     4    4     4
          law          152      1     1    1     1
          questioner   215     11    11   11    11
summary_q single      3016      6     6    2     2

In [11]:
# 약칭 표가 더한 값 중 틀린 것
for qid, (docs, qtype) in {**gold["multidoc"], **gold["summary_q"]}.items():
    old = set(ex_old.extract(text[qid]).get("committee_name", []))
    for v in cond[qid].get("committee_name", []):
        if v not in old and not any(v in values[d]["committee_name"] for d in docs):
            print(qtype, v)

single 국회운영위원회
single 예산결산특별위원회
single 예산결산특별위원회
single 행정안전위원회


## `year_ref`

발화율과, 정답 문서 연도에서 `year_ref`를 뺀 차이의 분포. 정답 문서가 여럿이면 문서마다 센다.

In [12]:
rows = []
for name, g in gold.items():
    for qid, (docs, qtype) in g.items():
        for y in cond[qid].get("year_ref", []):
            for d in docs:
                doc_year = min(values[d]["date"], key=len)
                rows.append({"set": name, "type": qtype, "qid": qid,
                             "diff": int(doc_year) - int(y)})
yr = pd.DataFrame(rows)
fired = {name: sum("year_ref" in cond[q] for q in g) / len(g) for name, g in gold.items()}
print("발화율", {k: round(v, 3) for k, v in fired.items()})
yr.assign(diff=yr["diff"].clip(-2, 2)).groupby(["set", "type", "diff"]).size().unstack(fill_value=0)

발화율 {'multidoc': 0.09, 'summary_q': 0.001}


diff                  -2  -1   0   1   2
set       type                          
multidoc  conf         0  31  18   0   0
          questioner   1  71  21   3   0
summary_q single       0   0   2   0   1

In [13]:
# 차이가 -1·0·+1 밖인 경우 몇 건(질의 일부)
for _, r in yr[yr["diff"].abs() > 1].drop_duplicates("qid").head(5).iterrows():
    m = re.search(r"\d{2,4}\s*(?:회계\s*)?[년연]도", text[r.qid])
    i = m.start() if m else 0
    print(r.set, r.type, r["diff"], "|", text[r.qid][max(0, i - 20): i + 30])

multidoc questioner -2 | 9년 4월 22일 예산결산특별위원회 1999년도 제1회 추가경정예산안 심사와 2000년 1
summary_q single 30 | 89년도 국가보안법 위반 작품은 소각되었는데, 왜 이 


## 두 자리 연도

In [14]:
two = {q: [m.group() for m in DATE.finditer(t) if m.group(2)] for q, t in text.items()}
two = {q: v for q, v in two.items() if v}
print("두 자리 연도가 date로 뽑힌 질의", len(two))
yref2 = sum(bool(re.search(r"(?<!\d)'?\d{2}\s*(?:회계\s*)?[년연]도", t)) for t in text.values())
print("두 자리 year_ref가 있는 질의", yref2)

두 자리 연도가 date로 뽑힌 질의 0
두 자리 year_ref가 있는 질의 2


## 관찰

- 위원회 약칭 표: G3에서 16개 질의(`conf` 4, `law` 1, `questioner` 11)에 위원회 값이 더해졌고 16개 모두 정답 문서 위원회와 맞았다. 16개 모두 S11 규칙으로는 위원회가 적중하지 않던 질의다. dev single-doc에서는 6개 질의에 더해졌고 2개가 맞았다
- `year_ref`: G3 발화율 9.0%(54개 질의), dev single-doc 0.1%다. 정답 문서 연도에서 `year_ref`를 뺀 차이는 G3 문서 쌍 146개 중 -1이 102개(70%), 0이 39개, +1이 3개, 그 밖이 2개다. `2019년도 예산안`을 2018년에 심사하는 것처럼 회의는 대개 언급한 연도의 전해에 열렸다. `year_ref`를 날짜와 같은 해로만 맞추면 대부분 놓친다. 맞추는 방식은 EXP-003에서 정한다
- 두 자리 연도: `'YY년` 형태로 날짜가 뽑힌 질의는 두 세트 모두 0개다. 두 자리 `year_ref`는 2개 질의에 있고, 하나는 회의와 30년 떨어진 과거 사건 언급이다
- 기존 필드에서 사라진 값은 없다(메인이 S11 조건 파일과 비교, 진행 기록)
